## Creacion de Training Pipeline

### By: Carlos Javier Palacios Sanchez

### Date: 06/09/2026

### Description:

Requerimiento
Crear un script en Python llamado `train_pipeline.py` que:

- Lea los features procesados desde el archivo del paso anterior.
- Realice la separación train/test, el entrenamiento y la evaluación con métricas apropiadas.
- Almacene el modelo entrenado y los resultados de evaluación.

El script debe poder ejecutarse de forma autónoma.

Puede utilizar como ejemplo:
https://github.com/JoseRZapata/air-quality-fti/blob/main/src/pipelines/training_pipeline/train-pipeline.py

Entregables:

- Script `train_pipeline.py` funcional.
- Pruebas unitarias que cubran lectura de datos, entrenamiento, generación de métricas
  y almacenamiento del modelo.

---

## Solución

| Entregable | Ruta |
|---|---|
| Script | `src/pipelines/training_pipeline/train_pipeline.py` |
| Pruebas unitarias | `tests/pipelines/training_pipeline/test_train_pipeline.py` |

### Dónde encaja en la cadena

```
data/01_raw/corazon.csv
        │  feature_pipeline.py   (transformaciones deterministas + validación)
        ▼
data/04_feature/corazon_features.parquet
        │  train_pipeline.py     (transformaciones ajustadas + modelo)
        ▼
data/06_models/    · modelo_corazon.joblib, modelo_corazon_completo.joblib
data/07_model_output/ · predicciones_test.csv
data/08_reporting/ · metricas_entrenamiento.csv, metricas_entrenamiento.json
```

### La decisión de diseño que une los dos pipelines

El feature pipeline dejó los valores faltantes **sin imputar**, a propósito. Esa
decisión se cobra aquí: la imputación y el escalado van dentro del `Pipeline` de
scikit-learn, que se ajusta **sólo con el conjunto de entrenamiento**.

Si se hubiera imputado en el paso anterior, la mediana se habría calculado sobre las
480 filas —incluidas las que hoy forman el test— y esa información se habría colado
en el entrenamiento. El modelo habría parecido mejor de lo que es, y la mejora habría
desaparecido en producción. Es el error más común y más silencioso del *machine
learning* aplicado: no falla nada, simplemente las métricas mienten.

## 1. Configuración

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import joblib
import pandas as pd


def localizar_raiz() -> Path:
    """Sube por el árbol de directorios hasta encontrar el pyproject.toml."""
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pyproject.toml").is_file():
            return candidato
    raise FileNotFoundError("No se encontró la raíz del proyecto")


RAIZ = localizar_raiz()
SCRIPT = RAIZ / "src" / "pipelines" / "training_pipeline" / "train_pipeline.py"
PRUEBAS = RAIZ / "tests" / "pipelines" / "training_pipeline"

FEATURES = RAIZ / "data" / "04_feature" / "corazon_features.parquet"
MODELO = RAIZ / "data" / "06_models" / "modelo_corazon.joblib"
ARTEFACTO = RAIZ / "data" / "06_models" / "modelo_corazon_completo.joblib"
PREDICCIONES = RAIZ / "data" / "07_model_output" / "predicciones_test.csv"
METRICAS = RAIZ / "data" / "08_reporting" / "metricas_entrenamiento.csv"
MANIFIESTO = RAIZ / "data" / "08_reporting" / "metricas_entrenamiento.json"

sys.path.insert(0, str(RAIZ / "src"))

from pipelines.training_pipeline import train_pipeline as tp  # noqa: E402

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print(f"Raíz del proyecto : {RAIZ}")
print(f"Script            : {SCRIPT.relative_to(RAIZ)}  (existe: {SCRIPT.is_file()})")
print(f"Features de entrada: {FEATURES.relative_to(RAIZ)}  (existe: {FEATURES.is_file()})")

Raíz del proyecto : /mnt/c/Users/KATANA/Heart_project
Script            : src/pipelines/training_pipeline/train_pipeline.py  (existe: True)
Features de entrada: data/04_feature/corazon_features.parquet  (existe: True)


## 2. Configuración del entrenamiento

Todas las decisiones del entrenamiento son constantes al inicio del script, no
números repartidos por el código.

In [2]:
print(f"Semilla única          : {tp.SEMILLA}")
print(f"Proporción de test     : {tp.PROPORCION_TEST:.0%}")
print(f"Particiones de la CV   : {tp.N_PARTICIONES}")
print(f"Métrica principal      : {tp.METRICA_PRINCIPAL}")
print(f"Umbral por defecto     : {tp.UMBRAL_DEFECTO}")
print(f"Modelo por defecto     : {tp.MODELO_POR_DEFECTO}")
print(f"\nModelos disponibles ({len(tp.MODELOS)}):")
for nombre, estimador in tp.MODELOS.items():
    print(f"  {nombre:<22} {type(estimador).__name__}")

Semilla única          : 42
Proporción de test     : 20%
Particiones de la CV   : 5
Métrica principal      : f1
Umbral por defecto     : 0.5
Modelo por defecto     : gradient_boosting

Modelos disponibles (3):
  gradient_boosting      HistGradientBoostingClassifier
  regresion_logistica    LogisticRegression
  random_forest          RandomForestClassifier


### Por qué F1 y no exactitud

El dataset está prácticamente equilibrado (≈48 % de enfermos), así que la exactitud no
es absurda, pero sigue siendo la métrica equivocada para decidir. Esconde el tipo de
error: un modelo con 84 % de exactitud puede estar fallando casi todo en una clase.

En un tamizaje cardíaco los dos errores no cuestan lo mismo. Un **falso negativo** es
un paciente enfermo al que se manda a casa; un **falso positivo** es una prueba
adicional. F1 obliga a mantener sensibilidad y precisión juntas, y se reportan además
ROC AUC y PR AUC porque evalúan el ordenamiento de probabilidades sin depender del
umbral elegido.

In [3]:
print("Pipeline que se entrena:\n")
pipeline = tp.construir_pipeline()
for nombre, paso in pipeline.named_steps.items():
    print(f"  {nombre:<12} -> {type(paso).__name__}")

print("\nLos dos primeros pasos aprenden de los datos (la mediana, la media y la")
print("desviación). Al estar dentro del Pipeline, cross_validate los reajusta en cada")
print("partición y nunca ven el pliegue de validación.")

Pipeline que se entrena:

  imputador    -> SimpleImputer
  escalador    -> StandardScaler
  modelo       -> HistGradientBoostingClassifier

Los dos primeros pasos aprenden de los datos (la mediana, la media y la
desviación). Al estar dentro del Pipeline, cross_validate los reajusta en cada
partición y nunca ven el pliegue de validación.


## 3. Ejecución autónoma

Se invoca el script como proceso independiente, sin argumentos, para comprobar el
requisito de autonomía.

In [4]:
ejecucion = subprocess.run(  # noqa: S603
    [sys.executable, str(SCRIPT)],
    capture_output=True,
    text=True,
    check=False,
)

print(ejecucion.stderr or ejecucion.stdout)
print(f"Código de salida: {ejecucion.returncode}")

17:02:45 | INFO     | === Training pipeline: inicio (modelo: gradient_boosting) ===
17:02:45 | INFO     | Features leídos desde /mnt/c/Users/KATANA/Heart_project/data/04_feature/corazon_features.parquet -> 480 filas x 27 columnas
17:02:45 | INFO     | [entrenamiento] esquema 'features': OK
17:02:45 | INFO     | Atributos: 26 | positivos: 47.9%
17:02:46 | INFO     | Train: 384 filas (47.9% positivos) | Test: 96 filas (47.9% positivos)
17:02:47 | INFO     | Validación cruzada (5 particiones): F1 = 0.7686 ± 0.0303
17:02:48 | INFO     | Umbral óptimo por F1 fuera de muestra: 0.31 (F1 = 0.8177)
17:02:48 | INFO     | Modelo entrenado sobre 384 filas
17:02:48 | INFO     | Test - F1: 0.8454 | sensibilidad: 0.8913 | ROC AUC: 0.9422
17:02:48 | INFO     | Modelo guardado: /mnt/c/Users/KATANA/Heart_project/data/06_models/modelo_corazon.joblib
17:02:49 | INFO     | Artefacto guardado: /mnt/c/Users/KATANA/Heart_project/data/06_models/modelo_corazon_completo.joblib
17:02:49 | INFO     | Predicciones 

## 4. Separación train/test

La separación es estratificada: mantiene la proporción de enfermos en ambos
conjuntos. Con 480 filas, una partición al azar puede dejar un test con una
prevalencia bastante distinta, y entonces las métricas dejan de ser comparables entre
ejecuciones.

In [5]:
atributos, objetivo = tp.leer_features(FEATURES)
x_train, x_test, y_train, y_test = tp.separar_train_test(atributos, objetivo)

reparto = pd.DataFrame(
    {
        "filas": [len(x_train), len(x_test), len(atributos)],
        "% del total": [
            len(x_train) / len(atributos),
            len(x_test) / len(atributos),
            1.0,
        ],
        "% enfermos": [y_train.mean(), y_test.mean(), objetivo.mean()],
    },
    index=["train", "test", "total"],
)
reparto.style.format({"% del total": "{:.1%}", "% enfermos": "{:.1%}"})

,filas,% del total,% enfermos
train,384,80.0%,47.9%
test,96,20.0%,47.9%
total,480,100.0%,47.9%


El conjunto de test no vuelve a tocarse hasta la evaluación final. Ninguna decisión
—ni el modelo, ni los hiperparámetros, ni el umbral— se toma mirándolo.

## 5. Resultados de la evaluación

La tabla compara cinco escenarios. La columna que importa para decidir es
`validacion_cruzada`; `test` es la medición final, tomada una sola vez.

In [6]:
metricas = pd.read_csv(METRICAS, index_col="metrica")
metricas

,entrenamiento,validacion_cruzada,test,test_umbral_optimo,base_trivial
metrica,,,,,
f1,0.9565,0.7686,0.8454,0.8491,0.0000
sensibilidad,0.9565,0.7611,0.8913,0.9783,0.0000
precision,0.9565,0.7840,0.8039,0.7500,0.0000
exactitud_balanceada,0.9583,0.7806,0.8457,0.8391,0.5000
exactitud,0.9583,0.7812,0.8438,0.8333,0.5208
roc_auc,0.9912,0.8935,0.9422,0.9422,0.5000
pr_auc,0.9900,0.8851,0.9428,0.9428,0.4792


In [7]:
manifiesto = json.loads(MANIFIESTO.read_text(encoding="utf-8"))
matriz = manifiesto["matriz_confusion_test"]

tabla_confusion = pd.DataFrame(
    [
        [matriz["verdaderos_negativos"], matriz["falsos_positivos"]],
        [matriz["falsos_negativos"], matriz["verdaderos_positivos"]],
    ],
    index=["real: sano", "real: enfermo"],
    columns=["predicho: sano", "predicho: enfermo"],
)

print("Matriz de confusión sobre el test:\n")
print(tabla_confusion.to_string())
print(f"\nFalsos negativos (enfermos no detectados): {matriz['falsos_negativos']}")
print(f"Falsos positivos (sanos alarmados)       : {matriz['falsos_positivos']}")
print("\nEn tamizaje clínico el falso negativo es el error caro: el paciente se va a")
print("casa sin diagnóstico. Por eso el umbral se calibra para bajarlo.")

Matriz de confusión sobre el test:

               predicho: sano  predicho: enfermo
real: sano                 40                 10
real: enfermo               5                 41

Falsos negativos (enfermos no detectados): 5
Falsos positivos (sanos alarmados)       : 10

En tamizaje clínico el falso negativo es el error caro: el paciente se va a
casa sin diagnóstico. Por eso el umbral se calibra para bajarlo.


### El umbral como hiperparámetro

El 0.5 por defecto no tiene nada de especial: es un convenio, no una decisión clínica.
El script barre umbrales sobre las probabilidades de `cross_val_predict` —para cada
fila, de un modelo que no la vio al entrenar— y elige el que maximiza F1. Hacer ese
barrido sobre el test habría contaminado la evaluación final.

In [8]:
print(f"Umbral por defecto : {manifiesto['umbral_defecto']}")
print(f"Umbral óptimo (CV) : {manifiesto['umbral_optimo']}\n")

comparacion = pd.DataFrame(
    {
        "umbral 0.50": manifiesto["metricas"]["test"],
        f"umbral {manifiesto['umbral_optimo']:.2f}": manifiesto["metricas"]["test_umbral_optimo"],
    }
).round(4)
comparacion["diferencia"] = (comparacion.iloc[:, 1] - comparacion.iloc[:, 0]).round(4)
comparacion

Umbral por defecto : 0.5
Umbral óptimo (CV) : 0.30999999999999994



,umbral 0.50,umbral 0.31,diferencia
f1,0.8454,0.8491,0.0037
sensibilidad,0.8913,0.9783,0.0870
precision,0.8039,0.7500,-0.0539
exactitud_balanceada,0.8457,0.8391,-0.0066
exactitud,0.8438,0.8333,-0.0105
roc_auc,0.9422,0.9422,0.0000
pr_auc,0.9428,0.9428,0.0000


El umbral más permisivo compra sensibilidad a costa de precisión: detecta más enfermos
y alarma a más sanos. Cuál conviene no lo decide la estadística, sino el coste
clínico de cada error; el script guarda ambos y deja la decisión documentada.

## 6. ¿Está aprendiendo algo?

Dos comprobaciones obligatorias antes de dar por bueno un modelo: que supere a la
línea base trivial y que no esté memorizando el entrenamiento.

In [9]:
f1_test = manifiesto["metricas"]["test"]["f1"]
f1_trivial = manifiesto["metricas"]["base_trivial"]["f1"]
f1_train = manifiesto["metricas"]["entrenamiento"]["f1"]
f1_cv = manifiesto["metricas"]["validacion_cruzada"]["f1"]

print(f"F1 base trivial (clase mayoritaria) : {f1_trivial:.4f}")
print(f"F1 validación cruzada               : {f1_cv:.4f}")
print(f"F1 test                             : {f1_test:.4f}")
print(f"F1 entrenamiento                    : {f1_train:.4f}")
print(f"\nBrecha train - test: {manifiesto['brecha_train_test_f1']:.4f}")

F1 base trivial (clase mayoritaria) : 0.0000
F1 validación cruzada               : 0.7686
F1 test                             : 0.8454
F1 entrenamiento                    : 0.9565

Brecha train - test: 0.1112


El modelo supera con holgura a la base trivial, así que sí está extrayendo señal de
los atributos.

La brecha entre entrenamiento y test es de unos 0.11 puntos de F1: hay algo de
sobreajuste, esperable en un boosting con 384 filas de entrenamiento, pero el modelo
generaliza. Vale la pena ser honesto con este número en lugar de esconderlo: es la
razón por la que el modelo lleva regularización (`l2_regularization`,
`min_samples_leaf`, `early_stopping`) en vez de los valores por defecto.

### Comparación entre los tres candidatos

El script permite reentrenar con cualquiera de los tres modelos. La comparación se
hace por **validación cruzada**, no por el test: con 96 filas de test, una diferencia
de 0.02 en F1 es ruido, no evidencia.

In [10]:
comparativa = []
for nombre in tp.MODELOS:
    pipeline_candidato = tp.construir_pipeline(nombre)
    resultado = tp.validar_cruzado(pipeline_candidato, x_train, y_train)
    comparativa.append({"modelo": nombre, **resultado})

tabla_modelos = (
    pd.DataFrame(comparativa).set_index("modelo").sort_values("f1", ascending=False).round(4)
)
tabla_modelos[["f1", "f1_std", "sensibilidad", "precision", "roc_auc"]]

,f1,f1_std,sensibilidad,precision,roc_auc
modelo,,,,,
regresion_logistica,0.8013,0.0369,0.7664,0.860,0.8980
random_forest,0.7976,0.0497,0.7883,0.815,0.9175
gradient_boosting,0.7686,0.0303,0.7611,0.784,0.8935


## 7. Almacenamiento del modelo

Se guardan dos versiones. `modelo_corazon.joblib` es sólo el pipeline, listo para
`predict`. `modelo_corazon_completo.joblib` incluye además los metadatos: un `.joblib`
suelto no dice con qué umbral se calibró ni con qué versión de scikit-learn se
entrenó, y ambos datos hacen falta para servirlo sin sorpresas.

In [11]:
modelo_recargado = joblib.load(MODELO)
artefacto = joblib.load(ARTEFACTO)

predicciones_ahora = modelo_recargado.predict(x_test)
print(f"El modelo recargado predice {len(predicciones_ahora)} filas sin error.\n")

print("Metadatos guardados junto al pipeline:")
for clave, valor in artefacto.items():
    if clave in {"pipeline", "metricas", "matriz_confusion_test"}:
        continue
    print(f"  {clave:<24} = {valor}")

El modelo recargado predice 96 filas sin error.

Metadatos guardados junto al pipeline:
  generado_en              = 2026-09-06T22:02:48+00:00
  fuente                   = /mnt/c/Users/KATANA/Heart_project/data/04_feature/corazon_features.parquet
  modelo                   = gradient_boosting
  n_filas                  = 480
  n_atributos              = 26
  n_train                  = 384
  n_test                   = 96
  semilla                  = 42
  metrica_principal        = f1
  umbral_defecto           = 0.5
  umbral_optimo            = 0.30999999999999994
  brecha_train_test_f1     = 0.1112
  version_sklearn          = 1.9.0


In [12]:
predicciones = pd.read_csv(PREDICCIONES, index_col="indice")
print(f"Predicciones guardadas: {predicciones.shape[0]} filas\n")
print("Los 5 casos con mayor probabilidad de enfermedad:")
predicciones.sort_values("probabilidad", ascending=False).head()

Predicciones guardadas: 96 filas

Los 5 casos con mayor probabilidad de enfermedad:


,real,probabilidad,prediccion_umbral_defecto,prediccion_umbral_optimo
indice,,,,
223,1,0.989937,1,1
76,1,0.989642,1,1
127,1,0.986066,1,1
364,1,0.985620,1,1
96,1,0.985265,1,1


Guardar las predicciones fila a fila permite auditar el modelo caso por caso: buscar
los falsos negativos concretos y mirar qué tenían esos pacientes. Un CSV de métricas
agregadas no permite esa conversación.

## 8. Pruebas unitarias

Cubren los cuatro puntos que pide el entregable —lectura de datos, entrenamiento,
generación de métricas y almacenamiento del modelo— más la ejecución de extremo a
extremo y los casos de fallo.

Los datos de prueba no son una tabla de features escrita a mano: se generan crudos
sintéticos con señal real y se pasan por el **feature pipeline de verdad**. Así las
pruebas usan el mismo contrato de datos que produce el paso anterior, y un cambio de
esquema las rompe, que es exactamente lo que queremos que pase.

In [13]:
pruebas = subprocess.run(  # noqa: S603
    [sys.executable, "-m", "pytest", str(PRUEBAS), "-v", "--no-header"],
    cwd=RAIZ,
    capture_output=True,
    text=True,
    check=False,
)

print(pruebas.stdout[-7000:])
print(f"Código de salida: {pruebas.returncode}")

============================= test session starts ==============================
collecting ... collected 33 items

tests/pipelines/training_pipeline/test_train_pipeline.py::test_leer_features_separa_atributos_y_objetivo PASSED [  3%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_leer_features_valida_el_esquema PASSED [  6%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_leer_features_falla_si_no_existe PASSED [  9%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_leer_features_falla_sin_objetivo PASSED [ 12%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_leer_features_falla_con_esquema_corrupto PASSED [ 15%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_separar_respeta_la_proporcion PASSED [ 18%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_separar_estratifica_la_clase PASSED [ 21%]
tests/pipelines/training_pipeline/test_train_pipeline.py::test_separar_es_reproducible PASSED [ 24%]
test

## 9. Conclusiones

El requerimiento queda cubierto en sus cuatro puntos:

1. **Lee los features procesados del paso anterior** — `data/04_feature/corazon_features.parquet`,
   revalidado contra el mismo esquema de Pandera que declaró el feature pipeline. El
   contrato se comprueba en los dos extremos, no sólo al escribir.
2. **Separación train/test, entrenamiento y evaluación con métricas apropiadas** —
   partición estratificada 80/20, validación cruzada de 5 particiones sobre el train,
   y siete métricas encabezadas por F1, con ROC AUC y PR AUC como contexto.
3. **Almacena el modelo entrenado y los resultados** — dos `.joblib` (pipeline y
   pipeline con metadatos), el CSV de métricas, el CSV de predicciones y el manifiesto
   JSON de la ejecución.
4. **Se ejecuta de forma autónoma** — sin argumentos, resolviendo las rutas del
   proyecto y devolviendo un código de salida apto para automatización.

Lo que vale la pena defender en la sustentación:

- **La frontera entre los dos pipelines no es organizativa, es metodológica.** Lo
  determinista va antes y se persiste; lo que aprende de los datos va después del
  `train_test_split`. Esa línea es la que evita el *data leakage*, y es la razón de
  que la tabla de features conserve los faltantes.
- **El test se usa una sola vez.** Modelo, hiperparámetros y umbral se eligen con
  validación cruzada sobre el entrenamiento. Un test consultado varias veces deja de
  ser una estimación del rendimiento futuro y se convierte en otro conjunto de
  validación, esta vez sin nada que lo respalde.